1.1  
Load data from web  
https://huggingface.co/datasets/FronkonGames/steam-games-dataset

In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("FronkonGames/steam-games-dataset")
df = dataset["train"].to_pandas()

print(df.shape)
print(df.head())
print(df.columns)

c:\Users\21424\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\21424\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\21424\.cache\huggingface\hub\datasets--FronkonGames--steam-games-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Py

(124146, 41)
     appID                                   name  release_date  \
0  2539430             Black Dragon Mage Playtest   Aug 1, 2023   
1   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016   
2  1034400      Mystery Solitaire The Black Raven   May 6, 2019   
3  3292190            버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024   
4  3631080                          Maze Quest VR  Apr 24, 2025   

  estimated_owners  peak_ccu  required_age  price  dlc_count  \
0            0 - 0         0             0   0.00          0   
1        0 - 20000         0             0   5.24          0   
2        0 - 20000         0             0   4.99          0   
3        0 - 20000         1             0   8.99          1   
4        0 - 20000         0             0   4.99          0   

                                detailed_description  \
0                                                      
1  Springtime, April: when the cherry trees come ...   
2  Immerse yourself in the most

1.2  
Save the lists which we need for Q1 and Q2

In [3]:
keep_columns = [
    # identifiers, only for tracking / error analysis, not model input
    "appID",
    "name",

    # structured metadata features
    "release_date",
    "required_age",
    "price",
    "dlc_count",
    "supported_languages",
    "full_audio_languages",
    "windows",
    "mac",
    "linux",
    "achievements",
    "developers",
    "publishers",
    "categories",
    "genres",
    "tags",

    # text features
    "short_description",
    "detailed_description",

    # label construction / analysis columns
    "estimated_owners",
    "peak_ccu",
    "positive",
    "negative",
    "recommendations",
    "user_score",
    "score_rank",
    "metacritic_score",
    "average_playtime_forever",
    "median_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_2weeks"
]

df = df[keep_columns].copy()

print(df.shape)
print(df.head())

(124146, 31)
     appID                                   name  release_date  required_age  \
0  2539430             Black Dragon Mage Playtest   Aug 1, 2023             0   
1   496350  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016             0   
2  1034400      Mystery Solitaire The Black Raven   May 6, 2019             0   
3  3292190            버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024             0   
4  3631080                          Maze Quest VR  Apr 24, 2025             0   

   price  dlc_count                 supported_languages full_audio_languages  \
0   0.00          0                                  []                   []   
1   5.24          0                           [English]                   []   
2   4.99          0  [English, French, German, Russian]                   []   
3   8.99          1                            [Korean]             [Korean]   
4   4.99          0                           [English]            [English]   

   windows    mac  

2.1  
Deal with missing data  
Missing value checking showed that the selected columns did not contain pandas NaN values. However, the dataset uses empty strings, empty lists, and zero values for some unavailable information. Therefore, we further checked empty strings in text fields and empty lists in list-based categorical fields. Empty list fields were handled later through feature construction, such as converting languages, genres, categories, and tags into count features.

In [ ]:
missing_summary = df.isnull().sum().sort_values(ascending=False)
print(missing_summary)


appID                       0
name                        0
release_date                0
required_age                0
price                       0
dlc_count                   0
supported_languages         0
full_audio_languages        0
windows                     0
mac                         0
linux                       0
achievements                0
developers                  0
publishers                  0
categories                  0
genres                      0
tags                        0
short_description           0
detailed_description        0
estimated_owners            0
peak_ccu                    0
positive                    0
negative                    0
recommendations             0
user_score                  0
score_rank                  0
metacritic_score            0
average_playtime_forever    0
median_playtime_forever     0
average_playtime_2weeks     0
median_playtime_2weeks      0
dtype: int64
release_date 0
short_description 8315
detailed_descriptio

2.2.1  
Check empty strings

In [6]:
string_columns = [
    "release_date",
    "short_description",
    "detailed_description",
    "estimated_owners",
    "score_rank"
]

for col in string_columns:
    if col in df.columns:
        empty_count = (df[col].astype(str).str.strip() == "").sum()
        print(col, empty_count)

release_date 0
short_description 8315
detailed_description 8428
estimated_owners 0
score_rank 124106


2.2.2  
The selected columns did not contain pandas NaN values. However, empty-string checking showed that 8,315 games had empty short descriptions and 8,428 games had empty detailed descriptions. These were kept as empty text values because the text vectorizer can handle empty descriptions. The score_rank field was empty for almost all games, so it was removed from the dataset.

In [9]:
df["short_description"] = df["short_description"].fillna("")
df["short_description"] = df["short_description"].astype(str)

df["detailed_description"] = df["detailed_description"].fillna("")
df["detailed_description"] = df["detailed_description"].astype(str)

df = df.drop(columns=["score_rank"], errors="ignore")


print(df.shape)

(124146, 30)


2.3.1  
Check empty list  
No empty lists were found in the selected list-based columns, including languages, developers, publishers, categories, genres, and tags.

In [10]:
list_columns = [
    "supported_languages",
    "full_audio_languages",
    "developers",
    "publishers",
    "categories",
    "genres",
    "tags"
]

for col in list_columns:
    if col in df.columns:
        empty_list_count = df[col].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
        print(col, empty_list_count)

supported_languages 0
full_audio_languages 0
developers 0
publishers 0
categories 0
genres 0
tags 0


3.1  
Remove irrelevant columns

In [12]:
# Drop clearly unusable columns
df = df.drop(columns=["score_rank"], errors="ignore")

# Keep appID and name for later error analysis, but do not use them as model features
print(df.shape)
print(df.columns.tolist())

(124146, 30)
['appID', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'supported_languages', 'full_audio_languages', 'windows', 'mac', 'linux', 'achievements', 'developers', 'publishers', 'categories', 'genres', 'tags', 'short_description', 'detailed_description', 'estimated_owners', 'peak_ccu', 'positive', 'negative', 'recommendations', 'user_score', 'metacritic_score', 'average_playtime_forever', 'median_playtime_forever', 'average_playtime_2weeks', 'median_playtime_2weeks']


4.1  
Deal with release_date

In [13]:
df["release_date_parsed"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date_parsed"].dt.year
df["release_year"] = df["release_year"].fillna(df["release_year"].median())

print(df[["release_date", "release_year"]].head())

   release_date  release_year
0   Aug 1, 2023          2023
1  Jul 29, 2016          2016
2   May 6, 2019          2019
3  Oct 31, 2024          2024
4  Apr 24, 2025          2025


4.2  
Deal with estimated_owners

In [14]:
def parse_estimated_owners(owner_range):
    owner_range = str(owner_range).replace(",", "").strip()
    
    if "-" not in owner_range:
        return None
    
    lower, upper = owner_range.split("-")
    lower = int(lower.strip())
    upper = int(upper.strip())
    
    return (lower + upper) / 2


df["estimated_owners_mid"] = df["estimated_owners"].apply(parse_estimated_owners)

print(df[["estimated_owners", "estimated_owners_mid"]].head(10))
print(df["estimated_owners_mid"].describe())

  estimated_owners  estimated_owners_mid
0            0 - 0                   0.0
1        0 - 20000               10000.0
2        0 - 20000               10000.0
3        0 - 20000               10000.0
4        0 - 20000               10000.0
5        0 - 20000               10000.0
6        0 - 20000               10000.0
7        0 - 20000               10000.0
8        0 - 20000               10000.0
9        0 - 20000               10000.0
count    1.241460e+05
mean     8.467425e+04
std      1.281719e+06
min      0.000000e+00
25%      1.000000e+04
50%      1.000000e+04
75%      1.000000e+04
max      1.500000e+08
Name: estimated_owners_mid, dtype: float64


5.1  
Define popular as top 25%  
For Q1, we used estimated_owners to construct the popularity label. Since estimated_owners is represented as a range, we converted each range into its midpoint. We then labelled games in the top 25% of estimated owners as popular, and the remaining games as not popular.

In [36]:
popularity_threshold = df["estimated_owners_mid"].quantile(0.75)

df["popular"] = (df["estimated_owners_mid"] > popularity_threshold).astype(int)

print("Popularity threshold:", popularity_threshold)
print(df["popular"].value_counts())
print(df["popular"].value_counts(normalize=True))

Popularity threshold: 10000.0
popular
0    98578
1    25568
Name: count, dtype: int64
popular
0    0.794049
1    0.205951
Name: proportion, dtype: float64


5.2.1  
Build highly_positive  
Get total_reviews and positive_ratio  
For Q2, we constructed a user review quality label using Steam positive and negative vote counts. We first calculated the total number of reviews and the positive review ratio. To reduce noise from games with very few reviews, we only kept games with at least 50 total reviews. Games with a positive review ratio of at least 80% were labelled as highly positive.

In [16]:
# Calculate total number of Steam user reviews
df["total_reviews"] = df["positive"] + df["negative"]

# Avoid division by zero
df["positive_ratio"] = df["positive"] / df["total_reviews"]
df.loc[df["total_reviews"] == 0, "positive_ratio"] = 0

print(df[["positive", "negative", "total_reviews", "positive_ratio"]].head(10))
print(df["total_reviews"].describe())
print(df["positive_ratio"].describe())

   positive  negative  total_reviews  positive_ratio
0         0         0              0        0.000000
1       252         3            255        0.988235
2        21         3             24        0.875000
3         0         0              0        0.000000
4         0         0              0        0.000000
5         0         0              0        0.000000
6       117        13            130        0.900000
7         0         0              0        0.000000
8        28         4             32        0.875000
9        19         6             25        0.760000
count    1.241460e+05
mean     1.199218e+03
std      3.243467e+04
min      0.000000e+00
25%      0.000000e+00
50%      7.000000e+00
75%      4.800000e+01
max      8.815087e+06
Name: total_reviews, dtype: float64
count    124146.000000
mean          0.506660
std           0.406881
min           0.000000
25%           0.000000
50%           0.652000
75%           0.883495
max           1.000000
Name: positive_ratio,

5.2.2  
Use the data which a game at least have 50 reviews

In [20]:
review_threshold = 50

df_q2 = df[df["total_reviews"] >= review_threshold].copy()

print("Original dataset size:", df.shape)
print("Q2 dataset size:", df_q2.shape)
print(df_q2["total_reviews"].describe())

Original dataset size: (124146, 36)
Q2 dataset size: (30621, 36)
count    3.062100e+04
mean     4.838605e+03
std      6.517402e+04
min      5.000000e+01
25%      9.800000e+01
50%      2.390000e+02
75%      9.010000e+02
max      8.815087e+06
Name: total_reviews, dtype: float64


5.2.3  
Define highly_positive  
We use 80% positove ratio as high positove ratio

In [21]:
positive_threshold = 0.80

df_q2["highly_positive"] = (df_q2["positive_ratio"] >= positive_threshold).astype(int)

print(df_q2["highly_positive"].value_counts())
print(df_q2["highly_positive"].value_counts(normalize=True))

highly_positive
1    16883
0    13738
Name: count, dtype: int64
highly_positive
1    0.551354
0    0.448646
Name: proportion, dtype: float64


5.2.4  
Check examples

In [22]:
print(
    df_q2[
        ["name", "positive", "negative", "total_reviews", "positive_ratio", "highly_positive"]
    ].head(20)
)

                                             name  positive  negative  \
1           Supipara - Chapter 1 Spring Has Come!       252         3   
6                              Armored Brigade II       117        13   
11                                        OMNIMUS        47         6   
12                             Fantasy General II      1025       210   
13                 Forests, Fields and Fortresses        64         2   
17                                        Redline        74        35   
21                                      Evertried        55        14   
26                      Bad North: Jotunn Edition     11367       831   
27                  Lucky Night: Texas Hold'em VR       144        35   
30      COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣        75        34   
32                                  Wizard Shrimp        95         2   
34                                     Epic Arena       196       104   
42         The Metronomicon: Slay The Dance Floor  

In [37]:
print("Full df shape:", df.shape)
print("Q2 df shape:", df_q2.shape)

print("\nQ1 popular distribution:")
print(df["popular"].value_counts(normalize=True))

print("\nQ2 highly_positive distribution:")
print(df_q2["highly_positive"].value_counts(normalize=True))

Full df shape: (124146, 36)
Q2 df shape: (30621, 37)

Q1 popular distribution:
popular
0    0.794049
1    0.205951
Name: proportion, dtype: float64

Q2 highly_positive distribution:
highly_positive
1    0.551354
0    0.448646
Name: proportion, dtype: float64


6.1  
Create basic feature columns

In [39]:
feature_columns = [
    "release_year",
    "required_age",
    "price",
    "dlc_count",
    "windows",
    "mac",
    "linux",
    "achievements",
    "supported_languages",
    "full_audio_languages",
    "developers",
    "publishers",
    "categories",
    "genres",
    "tags",
    "short_description",
    "detailed_description"
]

6.2  
Save Q1 data

In [40]:
q1_columns = [
    "appID",
    "name"
] + feature_columns + [
    "popular"
]

df_q1_model_data = df[q1_columns].copy()

print(df_q1_model_data.shape)
print(df_q1_model_data.head())

df_q1_model_data.to_csv("steam_q1_model_data.csv", index=False, encoding="utf-8")

(124146, 20)
     appID                                   name  release_year  required_age  \
0  2539430             Black Dragon Mage Playtest          2023             0   
1   496350  Supipara - Chapter 1 Spring Has Come!          2016             0   
2  1034400      Mystery Solitaire The Black Raven          2019             0   
3  3292190            버튜버 파라노이아 - Vtuber Paranoia          2024             0   
4  3631080                          Maze Quest VR          2025             0   

   price  dlc_count  windows    mac  linux  achievements  \
0   0.00          0     True  False  False             0   
1   5.24          0     True  False  False             0   
2   4.99          0     True   True  False             0   
3   8.99          1     True  False  False            19   
4   4.99          0     True  False  False             0   

                  supported_languages full_audio_languages  \
0                                  []                   []   
1              

6.3  
Save Q2 data

In [41]:
q2_columns = [
    "appID",
    "name"
] + feature_columns + [
    "highly_positive"
]

df_q2_model_data = df_q2[q2_columns].copy()

print(df_q2_model_data.shape)
print(df_q2_model_data.head())

df_q2_model_data.to_csv("steam_q2_model_data.csv", index=False, encoding="utf-8")

(30621, 20)
      appID                                   name  release_year  \
1    496350  Supipara - Chapter 1 Spring Has Come!          2016   
6   1934300                     Armored Brigade II          2025   
11  1108640                                OMNIMUS          2019   
12  1025440                     Fantasy General II          2019   
13  2113450         Forests, Fields and Fortresses          2023   

    required_age  price  dlc_count  windows    mac  linux  achievements  \
1              0   5.24          0     True  False  False             0   
6              0  35.99          1     True  False  False             0   
11             0   0.99          1     True   True   True             0   
12             0  13.99          4     True  False  False            77   
13             0   1.49          0     True   True  False            13   

                                  supported_languages full_audio_languages  \
1                                           [Engli

6.4  
Save the complete cleaning version

In [42]:
df.to_csv("steam_cleaned_with_q1_labels.csv", index=False, encoding="utf-8")
df_q2.to_csv("steam_cleaned_with_q2_labels.csv", index=False, encoding="utf-8")

The cleaned dataset was saved into separate files for the two prediction tasks. For Q1, steam_q1_model_data.csv contains selected metadata and text fields with the popularity label. For Q2, steam_q2_model_data.csv contains the same selected input fields with the highly_positive label. Identifier columns such as appID and name are kept for tracking and error analysis, but they should not be used as model input features.